## 1. Imports & Reading data

In [ ]:
import pandas as pd
import numpy as np  
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('ggplot')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

In [ ]:
# Loaded the source CSV into a pandas DataFrame.
# This file is the primary dataset for the EDA workflow.
# Setting display options helps inspect the full schema and many columns in one view.
df = pd.read_csv('Risk_Assessments.csv')

## 2. Understanding Data
- Dataframe shape
- head and tail
- dtypes
- describe

In [ ]:
print(f'Shape: {df.shape}')

In [ ]:
#Displaying the first and last 10 rows of the DataFrame to get an overview of the data.
display(df.head(10))
display(df.tail(10))

In [ ]:
df.columns.tolist()

In [ ]:
df.dtypes

In [ ]:
# Generating descriptive statistics for the DataFrame, including count, mean, std, min, 25%, 50%, 75%, and max for numerical columns, and unique, top, freq for categorical columns.
display(df.describe(include='all').T)

## 3. Data Prep
- Dropping irrelevant columns and rows
- Identifying duplicated columns
- Renaming Columns
- Feature Creation


In [ ]:
# Cleaning the DataFrame by removing duplicate rows and resetting the index to ensure a clean dataset for analysis.
# Duplicates can skew analysis and lead to incorrect conclusions, so it's important to handle them appropriately.
df = df.drop_duplicates().reset_index(drop=True)

# Cleaning column names by converting them to lowercase, replacing spaces and hyphens with underscores, and stripping any leading or trailing whitespace.
df.columns = [str(col).strip().lower().replace(' ', '_').replace('-', '_') for col in df.columns]

# Identifying and dropping unnecessary columns that may not contribute to the analysis, such as unnamed columns or those containing notes or remarks.
drop_cols = [col for col in df.columns if 'unnamed' in col.lower() or col.lower() in {'notes', 'remarks'}]
df = df.drop(columns=drop_cols)

# Adding a new column to the DataFrame that counts the number of missing values in each row, which can help identify rows with incomplete data and inform decisions about data cleaning or imputation.
df['missing_count'] = df.isnull().sum(axis=1)

if 'risk_score' in df.columns:
    # Creating a new column that categorizes the risk score into 'Low', 'Medium', and 'High' based on predefined thresholds.
    # This categorization can help in segmenting the data for further analysis or visualization.
    df['risk_category'] = pd.cut(df['risk_score'], 
                                 bins=[-np.inf, 25, 50, 75, np.inf], 
                                 labels=['Low', 'Moderate', 'High', 'Very High'],
                                 right=True
                                )


print(f'Cleaned shape: {df.shape}')
display(df.head())

## 4. Review Summary Stats
- Understand the range, central tendency, and scale of numeric variables
- Identify skewness, low variability, or anomalies

In [ ]:
# Identifying numeric columns in the DataFrame to generate descriptive statistics specifically for those columns, which can provide insights into the distribution and central tendency of the numerical data.
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
if numeric_cols:
    display(df[numeric_cols].describe().T)
else:
    print("No numeric columns found in the DataFrame.")



The sections above cover load, shape/dtype inspection, and basic profiling. The additions below fill the
gaps against the standard used for `Measurements.csv`: **grain integrity, missingness visualization,
categorical distribution, cross-field consistency, referral-rate diagnostics, and timestamp discipline** 
the checks that determine whether this risk table is safe to key into downstream models rather than just
descriptively summarized.

One note on the existing "Feature Creation" cell above: the `if 'risk_score' in df.columns:` branch never
fires for this schema — `Risk_Assessments.csv` carries risk as a categorical `risk_level`
(low/moderate/high/critical), not a numeric `risk_score`. It's left in place rather than deleted, but it's
effectively dead code for this dataset; Section 4 below works with the categorical field this table
actually has.


## 5. Grain integrity

Confirming the primary key and the natural key explicitly, rather than inferring uniqueness from
`describe()` output.

In [ ]:
# Primary key: risk_assessment_id should be unique
dupe_pk = df['risk_assessment_id'].duplicated().sum()
print(f"Duplicate risk_assessment_id values: {dupe_pk}")

# Natural key: this table appears to carry one risk assessment per screening.
# Confirm that explicitly rather than assuming it from row/column counts.
dupe_screening = df['screening_id'].duplicated().sum()
print(f"Duplicate screening_id values: {dupe_screening}")

if dupe_pk == 0 and dupe_screening == 0:
    print("\nGrain confirmed: one risk assessment per screening_id, keyed by risk_assessment_id.")
else:
    print("\nGrain violation found — investigate before this table is joined downstream.")

## 6. Missing data pattern (visual)

`describe(include='all')` showed `count == 144` for every column, i.e. no nulls, but a visual check is
worth doing on any table before trusting that, especially since this file loads from a raw CSV with no
schema enforcement at read time.

In [ ]:
import missingno as msno

fig, ax = plt.subplots(figsize=(9, 4))
msno.bar(df, ax=ax, color='#4C72B0')
plt.title('Column completeness')
plt.tight_layout()
plt.show()

## 7. Categorical distribution diagnostics

The substance of this table is categorical (`risk_level`, `risk_code`, `requires_referral`,
`assessment_source`, `rules_version`), these deserve deliberate distribution checks, not just whatever
surfaces incidentally from `describe()`.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# risk_level: the primary severity dimension
risk_level_order = ['low', 'moderate', 'high', 'critical']
sns.countplot(data=df, y='risk_level', order=risk_level_order, ax=axes[0, 0], color='#4C72B0')
axes[0, 0].set_title('risk_level distribution')

# risk_code: many distinct values, so show only the most frequent
top_codes = df['risk_code'].value_counts().head(10)
sns.barplot(x=top_codes.values, y=top_codes.index, ax=axes[0, 1], color='#DD8452')
axes[0, 1].set_title('Top 10 risk_code values')

# requires_referral: binary flag
sns.countplot(data=df, x='requires_referral', ax=axes[1, 0], color='#55A868')
axes[1, 0].set_title('requires_referral distribution')

# rules_version / assessment_source: confirm these are as constant as describe() suggested
sns.countplot(data=df, x='rules_version', ax=axes[1, 1], color='#C44E52')
axes[1, 1].set_title('rules_version distribution')

plt.tight_layout()
plt.show()

print("assessment_source unique values:", df['assessment_source'].unique().tolist())
print("rules_version unique values:", df['rules_version'].unique().tolist())

## 8. Cross-field consistency: `risk_code` vs `risk_level`

Every `risk_code` encodes a severity suffix (`_LOW`, `_MODERATE`, `_HIGH`, `_CRITICAL`). That suffix should
always agree with `risk_level`; a mismatch would mean the rules engine and the stored risk level
disagree, which is exactly the kind of bug that should be caught before a risk indicator ships.

In [ ]:
def extract_suffix(code):
    for level in ['LOW', 'MODERATE', 'HIGH', 'CRITICAL']:
        if code.endswith(level):
            return level.lower()
    return None

df['risk_code_suffix'] = df['risk_code'].apply(extract_suffix)

# Cross-tab: off-diagonal cells would indicate a mismatch between risk_code and risk_level
consistency = pd.crosstab(df['risk_level'], df['risk_code_suffix'])
display(consistency)

mismatches = df[df['risk_level'] != df['risk_code_suffix']]
print(f"\nRows where risk_level disagrees with the risk_code suffix: {len(mismatches)}")
if len(mismatches):
    display(mismatches[['risk_assessment_id', 'risk_code', 'risk_level', 'risk_code_suffix']])
else:
    print("None found — risk_code and risk_level are fully consistent in this extract.")

## 9. Referral rate by risk level

Sanity check on the rules engine's referral logic: referral rate should scale with severity, and it's
worth confirming there isn't a `high`/`critical` case slipping through without a referral flag.

In [ ]:
referral_by_level = df.groupby('risk_level')['requires_referral'].agg(['mean', 'sum', 'count'])
referral_by_level.columns = ['referral_rate', 'referral_count', 'total_count']
referral_by_level = referral_by_level.reindex(['low', 'moderate', 'high', 'critical'])
display(referral_by_level)

plt.figure(figsize=(7, 4))
sns.barplot(x=referral_by_level.index, y=referral_by_level['referral_rate'], color='#8172B2')
plt.ylim(0, 1.05)
plt.ylabel('referral rate')
plt.title('Referral rate by risk_level')
plt.tight_layout()
plt.show()

# Flag any high/critical case that did NOT get a referral flag
missed_referrals = df[(df['risk_level'].isin(['high', 'critical'])) & (~df['requires_referral'])]
print(f"high/critical rows missing a referral flag: {len(missed_referrals)}")
if len(missed_referrals):
    display(missed_referrals[['risk_assessment_id', 'risk_level', 'risk_code', 'requires_referral']])

## 10. Timestamp discipline

Same standard applied to `Measurements.csv`: confirm `assessed_at` parses cleanly and is consistently UTC
before it's used in any windowing or time-based logic.

In [ ]:
df['assessed_at_parsed'] = pd.to_datetime(df['assessed_at'], utc=True, errors='coerce')
n_unparseable = df['assessed_at_parsed'].isna().sum()
print(f"Unparseable timestamps: {n_unparseable}")
print(f"Range: {df['assessed_at_parsed'].min()}  ->  {df['assessed_at_parsed'].max()}")

non_utc_suffix = (~df['assessed_at'].str.endswith('Z')).sum()
print(f"Timestamps not using explicit 'Z' (UTC) suffix: {non_utc_suffix}")

## 11. Summary of findings

- **Grain:** confirmed one risk assessment per `screening_id`, keyed uniquely by `risk_assessment_id`; no
  duplicates found (Section 4).
- **Completeness:** all 8 source columns are fully populated; confirmed both numerically and visually
  (Section 5).
- **Categorical structure:** `risk_level` has 4 levels (low/moderate/high/critical); `assessment_source`
  and `rules_version` are constant across this extract (`rules_engine`, `P80-RULES-1.0`); worth confirming
  that's expected rather than an artifact of a single batch/date range (Section 6).
- **Cross-field consistency:** `risk_code` severity suffixes agree with `risk_level` for every row in this
  extract (Section 7) ; a useful invariant to keep testing as new rule versions ship.
- **Referral logic:** referral rate is 0% for low/moderate and 100% for high/critical in this extract, with
  no high/critical rows missing a referral flag (Section 8); again worth re-checking as data volume grows.
- **Timestamps:** `assessed_at` parses cleanly with a consistent UTC (`Z`) suffix (Section 9).
- **Known dead code:** the numeric `risk_score` branch in the "Feature Creation" cell doesn't apply to this
  schema and can be removed or replaced with logic built on `risk_level`/`risk_code_suffix` instead.

**Next steps:** this table joins cleanly to `Measurements.csv` on `screening_id` at a 1:1 grain; worth a
follow-up notebook validating that every screening in `Measurements.csv` has a corresponding row here (and
vice versa) before either feeds a combined risk-indicator view.
